## 2.1 理论计算题（一阶马尔可夫 + 拉普拉斯平滑）

### 已知：
- 给定字符序列："ababc"
- 词汇表：$\mathcal{V} = \{\mathrm{a}, \mathrm{b}, \mathrm{c}\}$
- 采用一阶马尔可夫模型估计 $p(x_t \mid x_{t-1})$
- 使用拉普拉斯平滑（加 1 平滑），计算时考虑所有可能转移（包括未出现的情况）。

### 1. 统计转移频次

序列 "ababc" 的相邻转移（一阶）如下：
- 位置 1→2：$\mathrm{a} \to \mathrm{b}$
- 位置 2→3：$\mathrm{b} \to \mathrm{a}$
- 位置 3→4：$\mathrm{a} \to \mathrm{b}$
- 位置 4→5：$\mathrm{b} \to \mathrm{c}$

统计得到以 $\mathrm{b}$ 为前驱的频次：
- $N_{b} = 2$（$\mathrm{b}$ 出现在位置 2 和 4）
- $N_{b, a} = 1$（$\mathrm{b} \to \mathrm{a}$ 出现 1 次）
- $N_{b, c} = 1$（$\mathrm{b} \to \mathrm{c}$ 出现 1 次）
- $N_{b, b} = 0$（$\mathrm{b} \to \mathrm{b}$ 未出现）

### 2. 计算条件概率（加 1 平滑）

拉普拉斯平滑公式（以 $\mathrm{b}$ 为条件）：
$$
p(x' \mid \mathrm{b}) = \frac{N_{b, x'} + 1}{N_b + |\mathcal{V}|}
$$
其中 $|\mathcal{V}| = 3$，$N_b = 2$，分母为 $2 + 3 = 5$。

- 计算 $p(\mathrm{a} \mid \mathrm{b})$：
$$
p(\mathrm{a} \mid \mathrm{b}) = (1+1) / (2+3) = 2/5
$$

- 计算 $p(\mathrm{c} \mid \mathrm{b})$：
$$
p(\mathrm{c} \mid \mathrm{b}) = (1+1) / (2+3) = 2/5
$$

（补充：$p(\mathrm{b} \mid \mathrm{b}) = (0+1) / (2+3) = 1/5$，三者概率之和为 $2/5 + 2/5 + 1/5 = 1$，满足归一性。）

最终答案：
$$
\boxed{p(\mathrm{a} \mid \mathrm{b}) = 2/5 \ (\text{即 } 0.4)}
$$
$$
\boxed{p(\mathrm{c} \mid \mathrm{b}) = 2/5 \ (\text{即 } 0.4)}
$$

In [7]:
# 2.2 编程题：文本预处理与滑动窗口生成
import re
from collections import Counter

def preprocess_text(text, n):
    """
    完成文本预处理：
    1. 小写化，去除标点（保留字母和空格）
    2. 按空格分词
    3. 按词频构建词汇表（频率降序，相同频率按字母升序），分配 ID 从 0 开始
    4. 滑动窗口生成长度为 n 的特征序列（词列表）和对应的下一个词标签
    返回：
        vocab_dict: {word: id}
        features:   list of list of str, 每个子列表长度为 n
        labels:     list of str, 与 features 一一对应
    """
    # 1. 小写并只保留字母和空格
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)   # 保留小写字母和空格
    # 2. 按空格分词（连续空格会被 split 自动处理）
    words = text.split()
    
    # 3. 构建词汇表（按频率排序）
    freq = Counter(words)
    # 按频率降序，频率相同按字母升序
    sorted_words = sorted(freq.items(), key=lambda x: (-x[1], x[0]))
    vocab_dict = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 4. 滑动窗口生成特征和标签
    features = []
    labels = []
    for i in range(len(words) - n):
        # 当前窗口 words[i:i+n] 作为特征
        features.append(words[i:i+n])
        # 下一个词作为标签（若有）
        labels.append(words[i+n])
    # 如果长度不够，则 features 和 labels 均为空
    return vocab_dict, features, labels

# 测试示例
text = "The time machine"
n = 2
vocab, feats, labs = preprocess_text(text, n)
print("Vocabulary:", vocab)
print("Features:", feats)
print("Labels:", labs)

Vocabulary: {'machine': 0, 'the': 1, 'time': 2}
Features: [['the', 'time']]
Labels: ['machine']


## 3.1 理论计算题（线性 RNN 的 BPTT 梯度推导）

### 已知：
- 线性 RNN（无偏置）定义：
  $$
  h_t = W_{hh} h_{t-1} + W_{hx} x_t, \quad o_t = W_{oh} h_t
  $$
- 损失函数为平方损失：
  $$
  L = \frac{1}{2} \sum_{t=1}^{T} (o_t - y_t)^2
  $$
- 要求推导损失对权重 $W_{hh}$ 的梯度表达式，并说明梯度消失或爆炸的条件。

### 1. 定义辅助变量并推导反向传播递推式

令 $\delta_t = \frac{\partial L}{\partial h_t}$（损失对第 $t$ 步隐藏状态的梯度）。

对于最后一个时间步 $t = T$：
$$
\delta_T = \frac{\partial L}{\partial o_T} \cdot \frac{\partial o_T}{\partial h_T} = (o_T - y_T) \cdot W_{oh}^T
$$

对于 $t < T$，梯度由当前时刻输出和下一时刻隐藏状态两部分组成：
$$
\delta_t = \frac{\partial L}{\partial o_t} \cdot \frac{\partial o_t}{\partial h_t} + \frac{\partial L}{\partial h_{t+1}} \cdot \frac{\partial h_{t+1}}{\partial h_t}
= (o_t - y_t) W_{oh}^T + \delta_{t+1} W_{hh}^T
$$

### 2. 推导对权重 $W_{hh}$ 的梯度

由于 $W_{hh}$ 在每个时间步都被复用，总梯度为所有时刻贡献之和：
$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L}{\partial h_t} \cdot \frac{\partial h_t}{\partial W_{hh}}
= \sum_{t=1}^{T} \delta_t \cdot h_{t-1}^T
$$
（其中 $h_{t-1}^T$ 表示列向量 $h_{t-1}$ 的转置，以保证矩阵维度匹配。）

最终梯度表达式为：
$$
\boxed{\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \delta_t \, h_{t-1}^T}
$$

### 3. 梯度消失与爆炸的条件

将 $\delta_t$ 沿时间轴展开，表达式中会包含 $W_{hh}$ 的幂次项（即 $(W_{hh}^T)^{T-t}$）。因此：
- 若 $W_{hh}$ 的谱半径（最大特征值绝对值）大于 1，梯度随 $T-t$ 指数增长，导致梯度爆炸。
- 若谱半径小于 1，梯度指数衰减，导致梯度消失。

$$
\boxed{\text{梯度爆炸条件：} \rho(W_{hh}) > 1 \quad;\quad \text{梯度消失条件：} \rho(W_{hh}) < 1}
$$

In [8]:
#3.2 编程题：简单 RNN 单元的前向与反向传播
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hh, W_hx, b_h):
    """
    RNN 单元前向传播（tanh 激活）
    x_t:   (batch_size, input_size)
    h_prev:(batch_size, hidden_size)
    W_hh:  (hidden_size, hidden_size)
    W_hx:  (input_size, hidden_size)   # 注意维度，便于矩阵乘法
    b_h:   (hidden_size,)
    返回:
        h_t: (batch_size, hidden_size)
        缓存: (x_t, h_prev, h_t, a) 用于反向
    """
    a = np.dot(h_prev, W_hh) + np.dot(x_t, W_hx) + b_h
    h_t = np.tanh(a)
    cache = (x_t, h_prev, h_t, a)
    return h_t, cache

def rnn_cell_backward(dh_next, cache):
    """
    反向传播，已知上游梯度 dh_next = ∂L/∂h_t
    计算 dx_t, dh_prev, dW_hh, dW_hx, db_h
    """
    x_t, h_prev, h_t, a = cache
    # tanh 导数: 1 - tanh^2(a)
    da = dh_next * (1 - np.tanh(a) ** 2)   # (batch_size, hidden_size)
    
    # 权重梯度：矩阵乘法形式
    dW_hh = np.dot(h_prev.T, da)            # (hidden_size, hidden_size)
    dW_hx = np.dot(x_t.T, da)               # (input_size, hidden_size)
    db_h = np.sum(da, axis=0)               # (hidden_size,)
    
    # 输入梯度
    dh_prev = np.dot(da, W_hh.T)            # (batch_size, hidden_size)
    dx_t = np.dot(da, W_hx.T)               # (batch_size, input_size)
    
    return dx_t, dh_prev, dW_hh, dW_hx, db_h

# 简单测试
batch_size, input_size, hidden_size = 2, 3, 4
x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hh = np.random.randn(hidden_size, hidden_size)
W_hx = np.random.randn(input_size, hidden_size)
b_h = np.random.randn(hidden_size)

h_t, cache = rnn_cell_forward(x_t, h_prev, W_hh, W_hx, b_h)
dh_next = np.random.randn(batch_size, hidden_size)
dx, dh_prev, dW_hh, dW_hx, db = rnn_cell_backward(dh_next, cache)
print("Gradients computed successfully.")

Gradients computed successfully.


## 4.1 理论计算题（深度双向 RNN 参数数量）

### 已知：
- 深度双向 RNN，共 $L$ 层
- 每层隐藏单元数为 $H$
- 输入维度为 $D$
- 输出维度（忽略，不计入参数）
- 要求计算模型参数总数（包含所有全连接层的权重和偏置）。

### 1. 计算第一层（输入层）参数

第一层输入维度为 $D$，输出维度为 $H$。双向 RNN 包含前向和后向两个独立方向，每个方向的参数为：
$$
D \times H \ (\text{输入到隐藏}) + H \times H \ (\text{隐藏到隐藏}) + H \ (\text{偏置}) = DH + H^2 + H
$$
两个方向合计：
$$
N_1 = 2 \times (DH + H^2 + H)
$$

### 2. 计算第 $l$ 层（$l \geq 2$）参数

从第 2 层开始，输入来自前一层双向输出的拼接，因此输入维度为 $2H$，输出维度为 $H$。单个方向的参数为：
$$
(2H) \times H + H \times H + H = 2H^2 + H^2 + H = 3H^2 + H
$$
两个方向合计：
$$
N_l = 2 \times (3H^2 + H)
$$
共有 $L-1$ 个这样的层。

### 3. 计算总参数数量

总参数数：
$$
\begin{aligned}
N_{\text{total}} &= N_1 + (L-1) \cdot N_l \\
&= 2(DH + H^2 + H) + (L-1) \cdot 2(3H^2 + H) \\
&= 2DH + 2H^2 + 2H + (L-1)(6H^2 + 2H) \\
&= 2DH + (2 + 6L - 6)H^2 + (2 + 2L - 2)H \\
&= 2DH + (6L - 4)H^2 + 2LH
\end{aligned}
$$

最终化简为：
$$
\boxed{N_{\text{total}} = 2H \left[ D + L + (3L - 2)H \right]}
$$

In [9]:
# 4.2 编程题：双向 RNN 编码器（使用 PyTorch）
import torch
import torch.nn as nn

def bidirectional_rnn_encoder(X, hidden_dim):
    """
    使用 torch.nn.RNN 实现双向编码器
    X:  (seq_len, batch, input_dim)
    hidden_dim: 每个方向的隐藏单元数
    返回:
        output: (seq_len, batch, 2*hidden_dim)  每个时间步拼接后的隐藏状态
        final:  (batch, 2*hidden_dim)           最终时间步的拼接（前向最后 + 后向最后）
    """
    seq_len, batch, input_dim = X.shape
    # 创建双向 RNN，单层
    rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim,
                 num_layers=1, batch_first=False, bidirectional=True)
    # 初始化隐藏状态（可选，默认全零）
    # 前向传播
    output, h_n = rnn(X)   # output: (seq_len, batch, 2*hidden_dim)
                           # h_n: (num_layers*2, batch, hidden_dim)
    # output 中每个时间步已经是前向+后向拼接
    # 最终时间步的拼接：取 output[-1, :, :]
    final = output[-1, :, :]   # (batch, 2*hidden_dim)
    return output, final

# 测试
seq_len, batch, input_dim = 5, 3, 10
hidden_dim = 8
X = torch.randn(seq_len, batch, input_dim)
output, final = bidirectional_rnn_encoder(X, hidden_dim)
print("Output shape:", output.shape)   # (5, 3, 16)
print("Final shape:", final.shape)     # (3, 16)

Output shape: torch.Size([5, 3, 16])
Final shape: torch.Size([3, 16])


## 5.1 理论计算题（Skip-gram 负采样损失函数）

### 已知：
- Skip-gram 模型中，给定中心词 $w_c$ 和上下文词 $w_o$
- 词向量分别为 $\mathbf{v}_c$（输入向量）和 $\mathbf{u}_o$（输出向量）
- 负采样从噪声分布中采样 $K$ 个负样本 $n_1, \dots, n_K$，对应词向量为 $\mathbf{u}_{n_k}$
- 要求推导损失函数（对数似然）表达式，并说明负样本采样方式。

### 1. 推导负采样损失函数（负对数似然）

负采样将多分类问题转化为二分类问题。对于正样本 $(w_c, w_o)$，我们希望最大化其共现概率；对于负样本，我们希望最小化其共现概率。

对数似然函数（最大化）为：
$$
\mathcal{L} = \log \sigma(\mathbf{v}_c^\top \mathbf{u}_o) + \sum_{k=1}^{K} \mathbb{E}_{n_k \sim P_n} \left[ \log \sigma(-\mathbf{v}_c^\top \mathbf{u}_{n_k}) \right]
$$
其中 $\sigma(x) = 1 / (1 + e^{-x})$。

实际优化中通常采用随机梯度下降最小化负对数似然，因此损失函数 $J$ 为：
$$
J = - \log \sigma(\mathbf{v}_c^\top \mathbf{u}_o) - \sum_{k=1}^{K} \log \sigma(-\mathbf{v}_c^\top \mathbf{u}_{n_k})
$$

完整目标函数（含期望形式）为：
$$
\boxed{
J = - \log \sigma(\mathbf{v}_c^\top \mathbf{u}_o) - \sum_{k=1}^{K} \mathbb{E}_{n_k \sim P_n} \left[ \log \sigma(-\mathbf{v}_c^\top \mathbf{u}_{n_k}) \right]
}
$$

### 2. 从噪声分布中采样负样本的方法

负样本并非从均匀分布中抽取，而是基于词频的分布。最常用的噪声分布是 Unigram 分布的 $3/4$ 次方：
$$
P_n(w) = \frac{\text{freq}(w)^{3/4}}{\sum_{v \in \mathcal{V}} \text{freq}(v)^{3/4}}
$$

这种设计可以降低高频词被过度采样的概率，使低频词也有机会被选为负样本，从而学习到更高质量的词向量表示。

$$
\boxed{\text{噪声采样分布：} P_n(w) \propto \text{freq}(w)^{3/4}}
$$


In [10]:
# 5.2 编程题：CBOW 前向传播与完整 softmax 损失
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, center_indices, W, W_out):
    """
    context_indices: list of lists, 每个子列表是 context_size 个上下文词的索引
                     形状 (batch_size, context_size)
    center_indices: 中心词索引，形状 (batch_size,)
    W:  (V, d) 输入嵌入矩阵
    W_out: (d, V) 输出权重矩阵
    返回:
        loss: 交叉熵损失（标量）
    """
    batch_size, context_size = context_indices.shape
    V, d = W.shape
    
    # 获取每个上下文词的嵌入向量 (batch_size, context_size, d)
    embeds = W[context_indices]   # 索引取行
    # 计算平均上下文向量 (batch_size, d)
    avg_embed = embeds.mean(dim=1)   # 对 context_size 维度平均
    
    # 计算得分 (batch_size, V)
    scores = torch.mm(avg_embed, W_out)   # (batch, d) @ (d, V) -> (batch, V)
    
    # 使用交叉熵损失，目标为 center_indices
    loss = F.cross_entropy(scores, center_indices)
    return loss

# 测试
V, d, context_size = 10, 5, 3
batch_size = 4
W = torch.randn(V, d, requires_grad=True)
W_out = torch.randn(d, V, requires_grad=True)
context = torch.randint(0, V, (batch_size, context_size))
center = torch.randint(0, V, (batch_size,))
loss = cbow_forward(context, center, W, W_out)
print("CBOW loss:", loss.item())

CBOW loss: 4.106278419494629


## 6.1 理论计算题（缩放点积注意力）

### 已知：
- 查询矩阵 $Q \in \mathbb{R}^{2 \times 4}$
- 键矩阵 $K \in \mathbb{R}^{3 \times 4}$
- 值矩阵 $V \in \mathbb{R}^{3 \times 5}$
- 缩放因子 $d_k = 4$，因此 $\sqrt{d_k} = 2$
- 要求计算缩放点积注意力输出矩阵，并写出中间步骤（得分矩阵、Softmax、加权求和）。

### 1. 计算未缩放得分矩阵 $S'$

得分矩阵为查询与键的点积：
$$
S' = Q K^\top \quad \in \mathbb{R}^{2 \times 3}
$$
其元素表示为：
$$
S'_{ij} = \sum_{l=1}^{4} Q_{i,l} \cdot K_{j,l} \quad (i=1,2;\ j=1,2,3)
$$

### 2. 缩放得分矩阵 $S$

缩放点积注意力使用 $\sqrt{d_k}$ 进行缩放，此处缩放因子为 $1 / 2$：
$$
S = S' / \sqrt{d_k} = S' / 2 \quad \in \mathbb{R}^{2 \times 3}
$$
即：
$$
S_{ij} = S'_{ij} / 2
$$

### 3. 按行计算 Softmax，得到注意力权重矩阵 $A$

对 $S$ 的每一行（每个查询）应用 Softmax 函数：
$$
A_{ij} = \frac{\exp(S_{ij})}{\sum_{m=1}^{3} \exp(S_{im})} \quad \in \mathbb{R}^{2 \times 3}
$$
此时矩阵 $A$ 的每一行元素之和为 1。

### 4. 加权求和得到最终输出矩阵 $O$

输出矩阵为注意力权重矩阵与值矩阵的乘积：
$$
O = A \cdot V \quad \in \mathbb{R}^{2 \times 5}
$$
其元素表示为：
$$
O_{i,p} = \sum_{j=1}^{3} A_{ij} \cdot V_{j,p} \quad (i=1,2;\ p=1,\dots,5)
$$

最终输出矩阵为：
$$
\boxed{\text{Output} = \text{softmax}\left( Q K^\top / \sqrt{4} \right) V}
$$
即：
$$
\boxed{O = \text{softmax}\left( Q K^\top / 2 \right) V \quad \in \mathbb{R}^{2 \times 5}}
$$

In [11]:
#6.2 编程题：多头注意力前向传播
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 2
        self.d_v = self.d_k

        # 线性投影层，分别用于 Q, K, V
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)  # 最终输出投影

    def forward(self, X):
        """
        X: (seq_len, batch, d_model)
        返回: (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape

        # 线性投影得到 Q, K, V，形状 (seq_len, batch, d_model)
        Q = self.W_Q(X)
        K = self.W_K(X)
        V = self.W_V(X)

        # 切分成多头: (seq_len, batch, num_heads, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k)
        K = K.view(seq_len, batch, self.num_heads, self.d_k)
        V = V.view(seq_len, batch, self.num_heads, self.d_v)

        # 交换维度便于计算: (batch, num_heads, seq_len, d_k)
        Q = Q.permute(1, 2, 0, 3)
        K = K.permute(1, 2, 0, 3)
        V = V.permute(1, 2, 0, 3)

        # 缩放点积注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        attn_weights = F.softmax(scores, dim=-1)   # (batch, num_heads, seq_len, seq_len)
        attn_output = torch.matmul(attn_weights, V) # (batch, num_heads, seq_len, d_v)

        # 合并多头: (batch, seq_len, num_heads * d_v) = (batch, seq_len, d_model)
        attn_output = attn_output.permute(2, 0, 1, 3).contiguous()
        attn_output = attn_output.view(seq_len, batch, self.d_model)

        # 最终线性层
        output = self.W_O(attn_output)  # (seq_len, batch, d_model)
        return output

# 测试
d_model = 4
num_heads = 2
seq_len, batch = 6, 3
X = torch.randn(seq_len, batch, d_model)
mha = MultiHeadAttention(d_model, num_heads)
out = mha(X)
print("Multi-Head Attention output shape:", out.shape)  # (6, 3, 4)

Multi-Head Attention output shape: torch.Size([6, 3, 4])
